# 📚 Bookcase Digitization - MASTER TRAINING NOTEBOOK
**Chú ý cực kỳ quan trọng:** Bạn PHẢI bật GPU trước khi chạy.
Vào `Runtime` -> `Change runtime type` -> Chọn `T4 GPU` -> `Save`.

In [ ]:
# 1. KIỂM TRA GPU (Nếu ra chữ CPU là phải làm lại bước trên)
import torch
if torch.cuda.is_available():
    print(f"✅ Đã bật GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ CHƯA BẬT GPU! HÃY DỪNG LẠI VÀ BẬT GPU THEO HƯỚNG DẪN TRÊN!")

### 2. Thiết lập Môi trường và Dữ liệu

In [ ]:
import os
import shutil

# Dọn dẹp sạch sẽ Colab
%cd /content/
if os.path.exists('bookcase-digitization'):
    shutil.rmtree('bookcase-digitization')
if os.path.exists('runs'):
    shutil.rmtree('runs')

# Clone dự án
!git clone https://github.com/pie-12/bookcase-digitization.git
%cd /content/bookcase-digitization

# Cài thư viện
!pip install -qr requirements.txt
!pip install -q craft-text-detector vietocr==0.3.5

# Clone YOLOv5
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5
    !pip install -qr yolov5/requirements.txt

# Tạo file cấu hình data.yaml tuyệt đối
with open('/content/bookcase-digitization/data.yaml', 'w') as f:
    f.write("""
train: /content/bookcase-digitization/dataset/images/train
val: /content/bookcase-digitization/dataset/images/train
nc: 6
names: ['Ten sach', 'Tac gia', 'Nha xuat ban', 'Tap', 'Nguoi dich', 'Tai ban']
""")
print("✅ Môi trường đã sẵn sàng!")

### 3. Huấn luyện Mô hình (Training)
Bắt đầu train với YOLOv5s để kiểm tra trước. Nếu muốn mô hình xịn hơn, hãy đổi `yolov5s.pt` thành `yolov5x6.pt` và `epochs` thành 100.

In [ ]:
%env WANDB_MODE=disabled

# Train 30 epoch với model nhỏ để có file best.pt nhanh nhất
!python yolov5/train.py \
    --img 640 \
    --batch 16 \
    --epochs 30 \
    --data /content/bookcase-digitization/data.yaml \
    --weights yolov5s.pt \
    --project /content/runs/train \
    --name bookcase_model

### 4. Kiểm tra Kết quả (Detect)

In [ ]:
import os
import glob
from IPython.display import Image, display

weights_path = "/content/runs/train/bookcase_model/weights/best.pt"
test_image = "/content/bookcase-digitization/data_test/1624445642850.jpg"

if os.path.exists(weights_path):
    print("✅ Đã tìm thấy file weights! Đang chạy Detect...")
    # Chạy Detect
    !python yolov5/detect.py --weights {weights_path} --img 640 --conf 0.25 --source {test_image} --project /content/runs/detect --name exp
    
    # Tìm và hiển thị ảnh kết quả mới nhất
    latest_detect = max(glob.glob('/content/runs/detect/exp*'), key=os.path.getmtime)
    result_img = os.path.join(latest_detect, os.path.basename(test_image))
    if os.path.exists(result_img):
        display(Image(filename=result_img))
else:
    print("❌ LỖI: Không tìm thấy file best.pt. Quá trình Train ở trên đã bị lỗi hoặc chưa chạy xong!")

### 5. Tải file Weights về máy

In [ ]:
from google.colab import files
if os.path.exists(weights_path):
    print("Đang chuẩn bị tải file best.pt...")
    files.download(weights_path)
else:
    print("Chưa có file để tải!")